In [1]:
from seismic import SeismicIndex

In [2]:
json_input_file = "/data1/knn_datasets/sparse_datasets/msmarco_v1_passage/cocondenser/json/docs_anserini.jsonl"

In [4]:
index = SeismicIndex.build(
    json_input_file,
    n_postings=3000,
    centroid_fraction=0.1,
    min_cluster_size=2,
    summary_energy=0.4, 
    max_fraction=4.0,
    load_content=True)


Building the index...
Configuration { pruning: GlobalThreshold { n_postings: 3000, max_fraction: 4.0 }, blocking: RandomKmeans { centroid_fraction: 0.1, min_cluster_size: 2, clustering_algorithm: RandomKmeansInvertedIndexApprox { doc_cut: 15 } }, summarization: EnergyPreserving { summary_energy: 0.4 }, knn: KnnConfiguration { nknn: 0, knn_path: None } }
Reading the collection..
Number of rows: 8841823
Elapsed time to read the collection: 686 secs
Distributing and pruning postings: 117 secs
Number of posting lists: 28679
Avg posting list length: 2405.02
Building summaries: 27 secs


In [5]:
import numpy as np
import json

file_path = "/data1/knn_datasets/sparse_datasets/msmarco_v1_passage/cocondenser/json/queries_anserini.tsv"

queries = []
with open(file_path, 'r') as f:
    for line in f:
        queries.append(json.loads(line))

MAX_TOKEN_LEN = 30
string_type  = f'U{MAX_TOKEN_LEN}'

queries_ids = np.array([q['id'] for q in queries], dtype=string_type)

query_components = []
query_values = []

for query in queries:
    vector = query['vector']
    query_components.append(np.array(list(vector.keys()), dtype=string_type))
    query_values.append(np.array(list(vector.values()), dtype=np.float32))

In [34]:
results = index.batch_search(
    queries_ids=queries_ids,
    query_components=query_components,
    query_values=query_values,
    k=10,
    query_cut=10,
    heap_factor=0.7,
    n_knn=0,
    sorted=True, #specified even if default value
    num_threads=1,
)

In [32]:
import ir_measures
import ir_datasets

# add your ir_dataset dataset string id below, e.g., "beir/quora/test"
ir_dataset_string = "msmarco-passage/dev/small"

ir_results = [ir_measures.ScoredDoc(query_id, doc_id, score) for r in results for (query_id, score, doc_id, _) in r]
qrels = ir_datasets.load(ir_dataset_string).qrels

In [44]:
res_pd = []
for r in results:
    for i, rr in enumerate(r):
        res_pd.append({
            "query_id": rr[0], 
            "doc_id": rr[2], 
            "score": rr[1], 
            "rank": i + 1
        })

res_pd = pd.DataFrame(res_pd)

In [46]:
import pandas as pd
df_qrels = pd.read_csv("/data1/knn_datasets/sparse_datasets/msmarco_v1_passage/qrels.dev.small.tsv", sep="\t", names=["query_id", "useless", "doc_id", "relevance"])



res_pd['doc_id'] = res_pd['doc_id'].astype(df_qrels.doc_id.dtype)


res_pd['query_id'] = res_pd['query_id'].astype(df_qrels.query_id.dtype)
metric = "RR@10"
ir_metric = ir_measures.parse_measure(metric)

metric_val = ir_measures.calc_aggregate([ir_metric], df_qrels, res_pd)[ir_metric]


print(f"Metric of the run ({ir_metric}): {round(metric_val, 4)}")


Metric of the run (RR@10): 0.3819


In [28]:
from ir_measures import *

measure_to_compute = "RR@10"
ir_measures.calc_aggregate([measure_to_compute], df_qrels, ir_results)

AttributeError: 'str' object has no attribute 'validate_params'